# 00_environment_and_data — adjust1

UCFace explicit settings take priority; otherwise FaceX-Zoo conventional training.
Target-only original sqrt margins; 112×112 inputs; full training lists and final batches.
VGGFace2: provided five-point landmarks + original FaceX-Zoo cropper.
TinyFace/SurvFace/SCface: user-approved resize-only, default bicubic; no face re-detection.
Existing shared_parent_v3 outputs are archived separately. Start with EXECUTE=False.


## 1. 사용자 설정
기본 5 seed와 네 head입니다. 데이터/완료 checkpoint를 수정하지 않습니다.

In [ ]:
from pathlib import Path
import sys, json
BASE_DIR = Path("F:/FaceX-Zoo")
SETTINGS_PATH = BASE_DIR / "fine-tune/data/research_v2_boundary/notebook_settings_mobilefacenet.json"
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
SEEDS = [8972, 8973, 8974, 8975, 8976]
HEADS = ["arcface", "adaface", "magface", "cosface"]
PARENT_SEED = 8972
PRETRAIN_EPOCHS = 40
PARENT_CHECKPOINT = None
PARENT_SHA256 = None
FINETUNE_DATASETS = ["survface", "tinyface", "scface"]
KEEP_RAW_RESULTS = True
DATA_INTEGRITY_MODE = "content_verified_reuse"  # full: repeat hash + decode
# Reuse decode only after every raw/prepared image hash matches; no stat-only trust.
DEVICE = "cuda:0"
NUM_WORKERS = 4
PREFETCH_FACTOR = 4
PERSISTENT_WORKERS = True
TORCH_THREADS = 2
LOSS_SYNC_BATCHES = 20
RECOVERY_MINUTES = 10
REUSE_COMPLETED = True
# Fresh adjust1 defaults; the variables below override saved settings and YAML.
defaults = resolve_settings(BASE_DIR)
DATASET = dict(defaults["dataset"])
BACKBONE = dict(defaults["backbone"])
# Resnet: type="Resnet", num_layers=50, mode="ir", feat_dim=1024.
# Use a separate settings file and parent checkpoint for each backbone.
BACKBONE.update(feat_dim=1024)
TRAINING = dict(defaults["training"])
TRAINING.update(save_freq=1, drop_last=False)
if BACKBONE["type"] in ("Resnet", "ResNet"):
    TRAINING["batch_size"] = 16  # GTX1080Ti starting choice; record and measure actual VRAM
PATHS = dict(defaults["paths"])
from modules.backbone_factory import backbone_slug
RUN_NAMESPACE = "research_v2_boundary_" + backbone_slug(BACKBONE)
for key, sub in (("checkpoint_dir", "checkpoints"), ("export_dir", "checkpoints/exported"), ("report_dir", "reports"), ("log_dir", "logs")):
    PATHS[key] = str(BASE_DIR / "fine-tune" / sub / RUN_NAMESPACE)

DATASET["alignment_report"] = str(BASE_DIR / "fine-tune/data/research_v2_boundary/preprocessed" / RUN_NAMESPACE / "vggface2/report.json")

from modules.finetune_protocol import research_profiles
FINETUNE_PROFILES = research_profiles(BASE_DIR, ronbun_root="C:/ronbun")
for name, profile in FINETUNE_PROFILES.items():
    profile["alignment_report"] = str(BASE_DIR / "fine-tune/data/research_v2_boundary/preprocessed" / RUN_NAMESPACE / name / "report.json")
HEAD_PARAMETERS = defaults["heads"]
PARENT_HEAD = dict(type="ArcFace", margin_arc=0.35, margin_am=0.0, scale=32.0)
# UCFace specifies augmentation types, not these numerical choices.
AUGMENTATION = dict(flip_probability=0.5, affine_probability=0.5,
    rotation_degrees=10.0, scale_range=[0.95, 1.05], border_mode="reflect", center="image_center")
# Resize-only is an approved experiment choice, not a confirmed UCFace recipe.
ALIGNMENT = defaults["alignment"]  # low_resolution.interpolation: bicubic (default) or bilinear
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
 "data_integrity":dict(mode=DATA_INTEGRITY_MODE),
 "project":dict(seed=SEEDS[0],seed_list=SEEDS,keep_raw_results=KEEP_RAW_RESULTS,device=DEVICE),
 "campaign":dict(heads=HEADS,reuse_completed=REUSE_COMPLETED,parent_seed=PARENT_SEED,
    pretrain_epochs=PRETRAIN_EPOCHS,parent_checkpoint=PARENT_CHECKPOINT,parent_sha256=PARENT_SHA256,
    finetune_datasets=FINETUNE_DATASETS),
 "finetune_profiles":FINETUNE_PROFILES,"dataset":DATASET,"backbone":BACKBONE,"heads":HEAD_PARAMETERS,
 "parent_head":PARENT_HEAD,"augmentation":AUGMENTATION,"alignment":ALIGNMENT,
 "training":{**TRAINING,"num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,
    "persistent_workers":PERSISTENT_WORKERS,"torch_threads":TORCH_THREADS,
    "recovery_minutes":RECOVERY_MINUTES,"loss_sync_batches":LOSS_SYNC_BATCHES},"paths":PATHS})


## 2. 설정 저장

In [ ]:
print(save_settings(SETTINGS_PATH, settings))

## 3. 실제 환경 확인

In [ ]:
from modules.notebook_workflow import inspect_environment
environment = inspect_environment(BASE_DIR, settings)
print(json.dumps(environment, ensure_ascii=False, indent=2))

## 4. 실행 계획

In [ ]:
for stage in ("pretrain","finetune","export"):
    plan=plan_matrix(settings,stage)
    print(stage,len(plan),[r["run_id"] for r in plan])
print("공통 부모 1개. 선택 데이터셋별 네 head × SEEDS. 공유 부모는 본 연구 선택입니다.")

## 5. 확인 범위

In [ ]:
print("부분 환경 점검입니다. 압축 전체 무결성·전체 이미지 decode·학습·FIQA 평가는 미실행입니다.")
print("안내:", BASE_DIR / "fine-tune/README.md")

새 연구 run은 target-only FP32 zero-radicand safeguard를 처음부터 사용합니다. 기존 parent/checkpoint는 새 정책으로 이어 학습하지 않습니다. 전처리 검증은 최초 hash+decode 후 현재 바이트 hash가 모두 일치할 때만 decode 결과를 재사용합니다. 검증 동안과 학습 중에는 데이터를 변경하지 않습니다.
분할은 기존 고정 후보를 먼저 평가하며, fit/safety 표본 수 분석은 ronbun에서 동일 모델·codec을 고정한 별도 단계로 수행합니다. FR/calibration 비율 비교는 필요성 검토 후 별도 분할·새 학습 namespace로 진행합니다.